In [ ]:
# https://my-json-server.typicode.com/st8324/repo/cafe
# 카페 매출 정보를 가져와서 DataFrame으로 변환하는 코드를 작성하세요.

In [ ]:
import requests
import pandas as pd

def get_cafe_revenue()->pd.DataFrame:
  
  url = 'https://my-json-server.typicode.com/st8324/repo/cafe'
    
  try:
        print(f"카페 매출 정보를 불러오는 중...")
        response = requests.get(url)
        response.raise_for_status()
        
        items = response.json()
        
        if not items:
            print("검색 결과가 없습니다.")
            return None
            
        order_list = []
        
        for item in items:
            order_list.append({
                '주문시간': item.get('주문시간', "미기입"),
                '메뉴': item.get('메뉴', "이름없음"),
                '카테고리': item.get('카테고리', "미분류"),
                '수량': item.get('수량', 0),
                '가격': item.get('가격', 0),
                '결제수단': item.get('결제수단', "기타"),
            })
            
        return pd.DataFrame(order_list)

  except Exception as e:
        print(f"오류 발생: {e}")
        return None
    
def save_to_csv(order_list):
    if order_list is not None:
        file_name = f'카페_매출_목록.csv'
        order_list.to_csv(file_name, index=False, encoding='utf-8-sig')
        print(f"{file_name} 이름으로 파일 저장이 완료되었습니다")

def clean_cafe_revenue(datas:pd.DataFrame)->pd.DataFrame:
  if datas is None or datas.empty:
        return datas
    
  result_df = datas.drop_duplicates()
  
  result_df = result_df.dropna(subset=['주문시간', '메뉴'])
    
  return result_df

if __name__ == "__main__":

    result_df = get_cafe_revenue()
    
    result_df = clean_cafe_revenue(result_df)
    
    if result_df is not None:
        print(result_df)
        save_to_csv(result_df)

In [ ]:
# 데이터를 활용한 예제
# 매출을 조회하기 위해 '결제액'을 추가
# 결제액은 가격 * 수량

result_df['결제액'] = (result_df['수량'] * result_df['가격'])

save_to_csv(result_df)

print(result_df)


In [ ]:
# 메뉴별 매출액을 조회(메뉴, 수량, 매출액)

def analyze_menu_revenue(df: pd.DataFrame):
    if df is None or df.empty:
        return None
      
    df['매출액'] = df['수량'] * df['가격']

    menu_stats = df.groupby('메뉴').agg({
        '수량': 'sum',
        '매출액': 'sum'
    }).reset_index()

    menu_stats = menu_stats.sort_values(by='매출액', ascending=False)
    
    return menu_stats

In [ ]:
if __name__ == "__main__":

    result_df = get_cafe_revenue()
    result_df = clean_cafe_revenue(result_df)
    
    if result_df is not None:
        print("\n=== 메뉴별 매출 분석 결과 ===")
        menu_revenue_df = analyze_menu_revenue(result_df)
        print(menu_revenue_df)



In [ ]:

# 카테고리별 매출 비율을 추가
# menu_revenue_df

def analyze_menu_revenue2(df: pd.DataFrame):
    if df is None or df.empty:
        return None
    
    df['매출액'] = df['수량'] * df['가격']
    df['총매출'] = df['매출액'].sum()
    df['매출비율'] = df['매출액'] / df['총매출'] * 100
    df['매출비율'] = df['매출비율'].round(1)

    menu_stats = df.groupby('메뉴').agg({
        '수량': 'sum',
        '매출액': 'sum',
        '매출비율': 'sum',
    }).reset_index()

    menu_stats = menu_stats.sort_values(by='매출비율', ascending=False)
    
    return menu_stats
  
if __name__ == "__main__":

    result_df = get_cafe_revenue()
    result_df = clean_cafe_revenue(result_df)
    
    if result_df is not None:
        print("\n=== 메뉴별 매출 분석 결과 ===")
        menu_revenue_df = analyze_menu_revenue2(result_df)
        print(menu_revenue_df)

In [ ]:
# 매출이 가장 높은 카테고리를 조회

sold_max = menu_revenue_df['매출액'].max()
print(sold_max)

In [ ]:
import pandas as pd
import requests

def get_cafe_revenue():
    url = 'https://my-json-server.typicode.com/st8324/repo/cafe'
    try:
        response = requests.get(url)
        response.raise_for_status()
        return pd.DataFrame(response.json())
    except Exception as e:
        print(f"데이터 로드 실패: {e}")
        return None

def get_time_slot(hour):
    if hour < 12: return "오전"
    elif hour < 17: return "오후"
    else: return "저녁"

result_df = get_cafe_revenue()

if result_df is not None:

    result_df['주문시간'] = pd.to_datetime(result_df['주문시간'])
    result_df['시간'] = result_df['주문시간'].dt.hour
    result_df['수량'] = result_df['수량'].astype(int)
    result_df['가격'] = result_df['가격'].astype(int)
    result_df['매출액'] = result_df['가격'] * result_df['수량']
    
    weekday_list = ['월', '화', '수', '목', '금', '토', '일']
    result_df['요일'] = result_df['주문시간'].dt.weekday.map(lambda x: weekday_list[x])
    
    result_df['시간대'] = result_df['시간'].apply(get_time_slot)
    
    result_df['요일'] = pd.Categorical(result_df['요일'], categories=weekday_list, ordered=True)
    
    weekday_stats = result_df.groupby('요일', observed=True).agg({
        '주문시간': 'count',
        '매출액': 'sum'
    }).rename(columns={'주문시간': '주문건수'}).reset_index()

    order = ['오전', '오후', '저녁']
    timeslot_revenue = result_df.groupby('시간대', observed=True)['매출액'].sum().reindex(order).reset_index()

    print("=== 요일별 현황 (주문건수 & 매출액) ===")
    print(weekday_stats)
    
    print("\n=== 시간대별 매출 현황 ===")
    print(timeslot_revenue)

    print("\n=== 전체 정제 데이터 ===")
    print(result_df[['주문시간', '요일', '메뉴', '매출액', '시간대']])

else:
    print("데이터가 없습니다.")
    

In [13]:
# 전체를 카페_매출.csv로 저장
result_df.to_csv('카페_매출.csv', index=False)

# 메뉴별 매출을 카페_메뉴_매출.csv로 저장
menu_revenue_df.to_csv('카페_메뉴_매출.csv')

# 카테고리별 매출을 카페_카테고리_매출.csv로 저장
result_df.to_csv('카페_카테고리_매출.csv')

# 시간대별 매출을 카페_시간_매출.csv로 저장
result_df.to_csv('카페_시간_매출.csv')

# 요일별 매출을 카페_요일_매출.csv로 저장
result_df.to_csv('카페_요일_매출.csv')

NameError: name 'menu_revenue_df' is not defined